# 03 — Compare on the untouched test split

Evaluates three model checkpoints on exactly the same 500 test images and identical noise realizations. Produces CSVs and summary JSON used by the web app recommendation.

## Step 1 — Setup and verify dataset

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd()
if not (ROOT / 'app').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'app').is_dir():
    raise FileNotFoundError('Open the complete iMAGE_Notebook_Edition folder in VS Code first.')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project:', ROOT)


In [ ]:
from app.experiments import dataset_counts, setup_device
print('Device:',setup_device());print(dataset_counts())


## Step 2 — Choose baseline or fine-tuned checkpoints

Setting `USE_FINETUNED=True` requires completed training for all three models. Otherwise the original published weights will load automatically from online sources. Both choices are legitimate experiments but **must be labeled**.

In [ ]:
from app.models import NAMES
from app.online import pretrained_path
USE_FINETUNED=False  # Set True when all three fine-tuned checkpoints exist
SIGMA=25
PATCH=256
SEED=42
LIMIT=0  # Must be 0 (all 500 images) for the final reported results
TILE=256
checkpoints={name:(ROOT/'weights'/f'{name}_finetuned.pth') if USE_FINETUNED else pretrained_path(name) for name in NAMES}
if USE_FINETUNED:
    for name,path in checkpoints.items():
        if not path.exists():raise FileNotFoundError(f'{name} checkpoint missing: {path}')
print('Checkpoints:',checkpoints)


## Step 3 — Evaluate each model sequentially. This can be computationally expensive.

All models use `test` images from the same CSV manifest; noise and crop coordinates are reproducible. The evaluation prints observed results, never fabricated benchmark scores.

In [ ]:
from app.experiments import evaluate_model
RUN_EVALUATION=False  # Change to True when ready
if RUN_EVALUATION:
    summaries=[]
    for name in NAMES:
        summaries.append(evaluate_model(name,checkpoint=checkpoints[name],
                                        patch=PATCH,sigma=SIGMA,seed=SEED,limit=LIMIT,tile=TILE))
else:
    print('Evaluation skipped. Change RUN_EVALUATION=True when ready.')


## Step 4 — Show comparison and charts

In [ ]:
from app.experiments import summaries_as_dataframe
import matplotlib.pyplot as plt
frame=summaries_as_dataframe()
if not frame.empty:
    display(frame[['model','number_of_test_images','psnr_db','ssim','mean_runtime_ms','parameters_m']])
    fig,ax=plt.subplots(figsize=(8,4))
    ax.bar(frame.model,frame.psnr_db)
    ax.set(ylabel='Mean PSNR (dB)',title='Same test images, noise level, and evaluation protocol')
    fig.tight_layout();fig.savefig(ROOT/'results'/'comparison_psnr.png',dpi=150);plt.show()
else: print('No saved results yet.')


## Step 5 — Deploy selected checkpoints for web inference

If `USE_FINETUNED=True`, copy chosen `.pth` files to `weights/{name}.pth`. The API will prefer those local weights over online baseline weights. Restart the server after changing checkpoints. Benchmark recommendations are only enabled when all three deployed checkpoints match full, comparable 500-image summaries.

In [ ]:
import shutil
DEPLOY_FINETUNED=False  # Change True only after evaluating the fine-tuned models
if DEPLOY_FINETUNED:
    if not USE_FINETUNED or LIMIT != 0:raise ValueError('Deploy only fully evaluated fine-tuned checkpoints.')
    for name,source in checkpoints.items():
        shutil.copy2(source,ROOT/'weights'/f'{name}.pth')
    print('Fine-tuned checkpoints deployed. Restart FastAPI with IMAGE_MODE=real.')
else:
    print('Keeping the current deployed checkpoints. No changes made.')


**Research documentation:** report data source, image split counts, source-image identities, noise formula/σ/seed, architecture and checkpoint provenance, training hyperparameters, training time, PSNR, SSIM, CPU/GPU device, and latency. If a checkpoint came from another noise distribution, report that limitation.